In [1]:
import sys
from pathlib import Path

current = Path.cwd()
while current != current.parent:
    if (current / 'src').exists() or (current / 'requirements.txt').exists():
        repo_root = current
        break
    current = current.parent
else:
    repo_root = Path.cwd()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repository root: {repo_root}")


Repository root: c:\Users\felip\OneDrive\Documentos\Estudos\TCC\TCC---Inventory-Forecasting


In [2]:
from src.data.preprocess_raw_data_by_month import preprocess_raw_data_by_month, debug_raw_directory

print("=== Checking data/raw directory ===")
debug_raw_directory()

result = preprocess_raw_data_by_month(verbose=True)
print(f"\nStatus: {result['status']}")
print(f"Pending files: {result['pending']}")
if result['summary']:
    print(f"Total records: {result['summary']['total_records']}")
    print(f"Total skipped: {result['summary']['total_skipped']}")


=== Checking data/raw directory ===

=== DEBUG: Checking c:\Users\felip\OneDrive\Documentos\Estudos\TCC\TCC---Inventory-Forecasting\data\raw ===
  [OK] Directory exists
  Total files: 26
  Files present:
    [XLSX] abril 24.xlsx
    [XLSX] abril 25.xlsx
    [XLSX] ago 24.xlsx
    [XLSX] ago 25.xlsx
    [XLSX] dez 24.xlsx
    [XLSX] dez 25.xlsx
    [XLSX] fev 24.xlsx
    [XLSX] fev 25.xlsx
    [XLSX] fev 26.xlsx
    [XLSX] jan 24.xlsx
    [XLSX] jan 25.xlsx
    [XLSX] jan 26.xlsx
    [XLSX] julho 24.xlsx
    [XLSX] julho 25.xlsx
    [XLSX] junho 24.xlsx
    [XLSX] junho 25.xlsx
    [XLSX] maio 24.xlsx
    [XLSX] maio 25.xlsx
    [XLSX] mar 24.xlsx
    [XLSX] mar 25.xlsx
    [XLSX] nov 24.xlsx
    [XLSX] nov 25.xlsx
    [XLSX] out 24.xlsx
    [XLSX] out 25.xlsx
    [XLSX] set 24.xlsx
    [XLSX] set 25.xlsx

26 arquivo(s) novo(s) para processar.

Processing: abril 24.xlsx
  [!] 201 registro(s) duplicado(s) ignorado(s).
  [OK] 11545 linhas adicionadas.
Processing: abril 25.xlsx
  [!] 182 r

## Pré-processamento: remoção de colunas e discretização

In [ ]:
from src.data.drop_columns import drop_columns

# Remove: operador, obs, icms_ret, arquivo_origem
caminho_reduzido = drop_columns(verbose=True)

In [ ]:
from src.data.discretize_fields import discretize_fields

# Converte pagamento, empresa e vendedor para códigos inteiros
# e salva o mapeamento de/para em src/utils/encodings.py
caminho_discretizado = discretize_fields(verbose=True)

## Análise Exploratória

In [ ]:
import pandas as pd
from src.utils.constants import PROCESSED_DIR
import os

df = pd.read_csv(
    os.path.join(PROCESSED_DIR, 'contas_a_receber_sem_colunas.csv'),
    sep=';', dtype=str, low_memory=False,
)
print(f"Shape: {df.shape}")
print(f"\nColunas: {list(df.columns)}")
df.head(3)

### Valores distintos: Empresa, Pagamento, Vendedor

In [ ]:
print("=== Empresa ===")
print(df['empresa'].value_counts().to_string())
print(f"\nTotal de empresas distintas: {df['empresa'].nunique()}")

In [ ]:
print("=== Pagamento ===")
print(df['pagamento'].value_counts().to_string())
print(f"\nTotal de formas de pagamento distintas: {df['pagamento'].nunique()}")

In [ ]:
print("=== Vendedor ===")
print(df['vendedor'].value_counts().to_string())
print(f"\nTotal de vendedores distintos: {df['vendedor'].nunique()}")

### Mapeamentos de/para (encodings)

In [ ]:
import importlib
import src.utils.encodings as enc
importlib.reload(enc)  # garante que usa o arquivo gerado nesta sessão

for campo, mapping in [
    ('pagamento', enc.PAGAMENTO_ENCODING),
    ('empresa',   enc.EMPRESA_ENCODING),
    ('vendedor',  enc.VENDEDOR_ENCODING),
]:
    print(f"=== {campo.upper()} ({len(mapping)} valores) ===")
    for valor, codigo in mapping.items():
        print(f"  {codigo:3d}  {valor}")
    print()

### Visualizações exploratórias

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df['empresa'].value_counts().plot(kind='barh', ax=axes[0])
axes[0].set_title('Registros por Empresa')
axes[0].set_xlabel('Quantidade')
axes[0].invert_yaxis()

df['pagamento'].value_counts().head(15).plot(kind='barh', ax=axes[1])
axes[1].set_title('Formas de Pagamento (top 15)')
axes[1].set_xlabel('Quantidade')
axes[1].invert_yaxis()

df['vendedor'].value_counts().head(15).plot(kind='barh', ax=axes[2])
axes[2].set_title('Vendedores com mais registros (top 15)')
axes[2].set_xlabel('Quantidade')
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
df_disc = pd.read_csv(
    os.path.join(PROCESSED_DIR, 'contas_a_receber_discretizado.csv'),
    sep=';', low_memory=False,
)
print(f"Shape: {df_disc.shape}")
print(f"\nTipos dos campos discretizados:")
print(df_disc[['pagamento', 'empresa', 'vendedor']].dtypes)
print(f"\nAmostra — colunas-chave:")
df_disc[['doc_tipo', 'doc_numero', 'pagamento', 'empresa', 'vendedor', 'produto_cod', 'tot_liq']].head(5)